<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/Plantdoc_VGG16_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/pratikkayal/PlantDoc-Dataset.git
!mv PlantDoc-Dataset /content/PlantDoc
!ls /content/PlantDoc

Cloning into 'PlantDoc-Dataset'...
remote: Enumerating objects: 2670, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 2670 (delta 22), reused 22 (delta 22), pack-reused 2635 (from 1)
Receiving objects: 100% (2670/2670), 932.92 MiB | 38.16 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Updating files: 100% (2581/2581), done.
LICENSE.txt  PlantDoc_Examples.png  README.md  test  train


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import VGG16, vgg16
from tensorflow.keras.preprocessing import image_dataset_from_directory
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize

In [ ]:
img_size = (224, 224)
batch_size = 32

train_ds_full = image_dataset_from_directory(
    "/content/PlantDoc/train",
    image_size=img_size,
    batch_size=batch_size
)

test_ds = image_dataset_from_directory(
    "/content/PlantDoc/test",
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds_full.class_names
num_classes = len(class_names)

# Split train into train + val (20%)
val_batches = tf.data.experimental.cardinality(train_ds_full) // 5
val_ds = train_ds_full.take(val_batches)
train_ds = train_ds_full.skip(val_batches)

print("Classes:", class_names)
print("Num Classes:", num_classes)

Found 2342 files belonging to 28 classes.
Found 236 files belonging to 27 classes.
Classes: ['Apple Scab Leaf', 'Apple leaf', 'Apple rust leaf', 'Bell_pepper leaf', 'Bell_pepper leaf spot', 'Blueberry leaf', 'Cherry leaf', 'Corn Gray leaf spot', 'Corn leaf blight', 'Corn rust leaf', 'Peach leaf', 'Potato leaf early blight', 'Potato leaf late blight', 'Raspberry leaf', 'Soyabean leaf', 'Squash Powdery mildew leaf', 'Strawberry leaf', 'Tomato Early blight leaf', 'Tomato Septoria leaf spot', 'Tomato leaf', 'Tomato leaf bacterial spot', 'Tomato leaf late blight', 'Tomato leaf mosaic virus', 'Tomato leaf yellow virus', 'Tomato mold leaf', 'Tomato two spotted spider mites leaf', 'grape leaf', 'grape leaf black rot']
Num Classes: 28


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

# 🔧 Add VGG16 preprocessing
preprocess_input = vgg16.preprocess_input

train_ds = train_ds.map(lambda x, y: (preprocess_input(data_augmentation(x)), y))
val_ds   = val_ds.map(lambda x, y: (preprocess_input(x), y))
test_ds  = test_ds.map(lambda x, y: (preprocess_input(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

In [ ]:
base_model = VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

def one_hot_map(images, labels):
    return images, tf.one_hot(labels, depth=num_classes)

train_ds_onehot = train_ds.map(one_hot_map)
val_ds_onehot   = val_ds.map(one_hot_map)
test_ds_onehot  = test_ds.map(one_hot_map)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
history = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=15
)

Epoch 1/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 70s 803ms/step - accuracy: 0.0602 - loss: 8.6630 - val_accuracy: 0.2500 - val_loss: 3.2296
Epoch 2/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 48s 734ms/step - accuracy: 0.1203 - loss: 4.2532 - val_accuracy: 0.2321 - val_loss: 3.3008
Epoch 3/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 82s 737ms/step - accuracy: 0.1518 - loss: 3.6010 - val_accuracy: 0.2790 - val_loss: 3.2407
Epoch 4/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 82s 727ms/step - accuracy: 0.1800 - loss: 3.4111 - val_accuracy: 0.2790 - val_loss: 3.1390
Epoch 5/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 82s 722ms/step - accuracy: 0.1976 - loss: 3.3333 - val_accuracy: 0.2790 - val_loss: 3.0513
Epoch 6/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 82s 736ms/step - accuracy: 0.1922 - loss: 3.3553 - val_accuracy: 0.3125 - val_loss: 2.9685
Epoch 7/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 48s 744ms/step - accuracy: 0.2084 - loss: 3.2327 - val_accuracy: 0.3571 - val_loss: 2.9253
Epoch 8/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 82s 743ms/step - accuracy: 0.2652 - loss: 3.0785 - val_accu

In [ ]:
base_model.trainable = True
fine_tune_at = 15   # keep first 15 layers frozen
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=40,
    initial_epoch=15
)

Epoch 16/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 58s 766ms/step - accuracy: 0.3192 - loss: 2.8468 - val_accuracy: 0.4174 - val_loss: 2.5746
Epoch 17/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 75s 736ms/step - accuracy: 0.3399 - loss: 2.7670 - val_accuracy: 0.4598 - val_loss: 2.4446
Epoch 18/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 50s 752ms/step - accuracy: 0.3604 - loss: 2.7053 - val_accuracy: 0.4866 - val_loss: 2.4215
Epoch 19/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 49s 744ms/step - accuracy: 0.3766 - loss: 2.6777 - val_accuracy: 0.4732 - val_loss: 2.4096
Epoch 20/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 82s 753ms/step - accuracy: 0.4090 - loss: 2.6189 - val_accuracy: 0.4688 - val_loss: 2.3936
Epoch 21/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 81s 749ms/step - accuracy: 0.4098 - loss: 2.5756 - val_accuracy: 0.4754 - val_loss: 2.3574
Epoch 22/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 87s 831ms/step - accuracy: 0.4116 - loss: 2.5709 - val_accuracy: 0.4844 - val_loss: 2.3549
Epoch 23/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 77s 745ms/step - accuracy: 0.4023 - loss: 2.5684 - 

In [ ]:
TEMPERATURE = 2.0

def predict_with_temperature(model, dataset, T=1.0):
    all_probs, all_labels = [], []
    for images, labels in dataset:
        images = tf.cast(images, tf.float32)
        logits = model(images, training=False)
        scaled_logits = logits / T
        probs = tf.nn.softmax(scaled_logits).numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_probs)

y_true, y_probs = predict_with_temperature(model, test_ds_onehot, T=TEMPERATURE)
y_pred = np.argmax(y_probs, axis=1)

# Convert one-hot labels to integers
y_true_int = np.argmax(y_true, axis=1)

# 🔧 Safe AUC computation
present_classes = np.unique(y_true_int)
if len(present_classes) > 1:
    y_true_bin_present = label_binarize(y_true_int, classes=present_classes)
    y_probs_present = y_probs[:, present_classes]
    auc = roc_auc_score(
        y_true_bin_present,
        y_probs_present,
        average="macro",
        multi_class="ovr"
    )
else:
    print("⚠️ Only one class present in test set, AUC not defined.")
    auc = np.nan

# Other metrics
accuracy = accuracy_score(y_true_int, y_pred)
precision = precision_score(y_true_int, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true_int, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true_int, y_pred, average="weighted", zero_division=0)
kappa = cohen_kappa_score(y_true_int, y_pred)

print("\n Evaluation Metrics After Calibration:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Kappa    : {kappa:.4f}")


 Evaluation Metrics After Calibration:
Accuracy : 0.4407
Precision: 0.3779
Recall   : 0.4407
F1 Score : 0.3801
AUC      : 0.9181
Kappa    : 0.4185
